# N145 · 高级DP优化及其成立条件

**目标：** 先验证决策单调或线性结构再使用优化。

**先修：** N108, N107, N137, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Monge/四边形不等式；分治优化；Knuth范围；凸包优化；Li Chao树；整数比较。

## 从问题到算法

高级优化首先是一个有条件的定理，而不是能替换所有双重循环的模板。分段DP为dp[g][r]=min_l(dp[g−1][l]+cost[l][r])。若代价满足Monge不等式，最左最优切点随r不减，才可递归缩小搜索范围。另一类转移dp[i]=min_j(dp[j]+b[j]a[i])可看成查询一组直线的下包络；Li Chao不要求斜率或查询点单调。

## 最小实现

**本章接口：** `partition_dp_quadratic(cost, k)`、`divide_conquer_dp(cost, k)`、`LiChaoMin.add_line/query`、`cht_dp_reference(a, b)`

In [1]:
def _cost_size(cost):
    n=len(cost)-1
    if n<0 or any(len(row)!=n+1 for row in cost): raise ValueError('cost must be an (n+1)x(n+1) matrix')
    return n

def partition_dp_quadratic(cost,k):
    n=_cost_size(cost)
    if k<0: raise ValueError('nonnegative segment count required')
    if k>n: return float('inf')
    prev=[0]+[float('inf')]*n
    for groups in range(1,k+1):
        dp=[float('inf')]*(n+1)
        for r in range(groups,n+1): dp[r]=min(prev[l]+cost[l][r] for l in range(groups-1,r))
        prev=dp
    return prev[n]

def _check_monge(cost):
    n=_cost_size(cost)
    for l in range(n):
        for r in range(l+1,n+1):
            x=cost[l][r]
            if x!=x or x in (float('inf'),float('-inf')): raise ValueError('finite segment costs required for this check')
    for l in range(n-1):
        for r in range(l+2,n):
            if cost[l][r]+cost[l+1][r+1]>cost[l][r+1]+cost[l+1][r]: raise ValueError('Monge inequality failed; use the quadratic reference')

def divide_conquer_dp(cost,k,*,check=True):
    n=_cost_size(cost)
    if k<0: raise ValueError('nonnegative segment count required')
    if k>n: return float('inf')
    if check: _check_monge(cost)
    prev=[0]+[float('inf')]*n
    for groups in range(1,k+1):
        dp=[float('inf')]*(n+1)
        def compute(left,right,opt_left,opt_right):
            if left>right: return
            mid=(left+right)//2; best=float('inf'); split=opt_left
            for l in range(opt_left,min(mid-1,opt_right)+1):
                value=prev[l]+cost[l][mid]
                if value<best: best=value; split=l
            dp[mid]=best
            compute(left,mid-1,opt_left,split); compute(mid+1,right,split,opt_right)
        compute(groups,n,groups-1,n-1); prev=dp
    return prev[n]

class _LineNode:
    def __init__(self,line): self.line=line; self.left=self.right=None

def _evaluate(line,x): return line[0]*x+line[1]

class LiChaoMin:
    def __init__(self,left,right):
        if left>right: raise ValueError('nonempty inclusive integer domain required')
        self.left=left; self.right=right; self.root=None
    def add_line(self,slope,intercept):
        def insert(node,l,r,line):
            if node is None: return _LineNode(line)
            mid=(l+r)//2
            if _evaluate(line,mid)<_evaluate(node.line,mid): line,node.line=node.line,line
            if l==r: return node
            if _evaluate(line,l)<_evaluate(node.line,l): node.left=insert(node.left,l,mid,line)
            elif _evaluate(line,r)<_evaluate(node.line,r): node.right=insert(node.right,mid+1,r,line)
            return node
        self.root=insert(self.root,self.left,self.right,(slope,intercept))
    def query(self,x):
        if not self.left<=x<=self.right: raise ValueError('query outside fixed integer domain')
        node=self.root; l,r=self.left,self.right; answer=float('inf')
        while node:
            answer=min(answer,_evaluate(node.line,x)); mid=(l+r)//2
            if l==r: break
            if x<=mid: node=node.left; r=mid
            else: node=node.right; l=mid+1
        return answer

def cht_dp_reference(a,b):
    if len(a)!=len(b): raise ValueError('equal lengths required')
    if not a: return []
    dp=[0]
    for i in range(1,len(a)): dp.append(min(dp[j]+b[j]*a[i] for j in range(i)))
    return dp

def cht_dp_li_chao(a,b):
    if len(a)!=len(b): raise ValueError('equal lengths required')
    if not a: return []
    tree=LiChaoMin(min(a),max(a)); tree.add_line(b[0],0); dp=[0]
    for i in range(1,len(a)): dp.append(tree.query(a[i])); tree.add_line(b[i],dp[-1])
    return dp

## 正确性、前提与复杂度

Monge条件C(a,c)+C(b,d)≤C(a,d)+C(b,c)在加上每行dp[g−1][l]后仍成立，最左最小值位置因此随列单调。这里对有效三角域检查相邻四边形，它们可望远镜相加得到更大矩形不等式。非负数组的区间和平方差为−2·左外段和·右外段和≤0。Li Chao在中点保留更优直线，另一条线只可能在一侧重新变优；沿查询路径取最小值即可。Knuth还需不同的区间递推和夹逼条件，不能由本章Monge检查自动推出。

**代价：** 平方参照O(kn²)；已满足单调条件的分治部分O(kn log n)，但默认Monge校验和输入矩阵本身仍为O(n²)，必须计入总成本。Li Chao在整数域宽度C上每次插入/查询O(log C)，不计算浮点交点；精确整数乘法位复杂度另计。check=False仅可用于已独立证明的代价，不能用来绕过失败测试。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,3,4,2]; prefix=[0]
for x in a: prefix.append(prefix[-1]+x)
cost=[[0]*(len(a)+1) for _ in range(len(a)+1)]
for l in range(len(a)):
    for r in range(l+1,len(a)+1): cost[l][r]=(prefix[r]-prefix[l])**2
show_table(['段数','平方版','分治版'],[(k,partition_dp_quadratic(cost,k),divide_conquer_dp(cost,k)) for k in range(1,len(a)+1)])
lines=[(2,3),(-1,8),(2,1)]; tree=LiChaoMin(-4,6)
for line in lines: tree.add_line(*line)
show_table(['x','各直线值','最小值'],[(x,[_evaluate(line,x) for line in lines],tree.query(x)) for x in range(-4,7)])

段数,平方版,分治版
1,144,144
2,72,72
3,54,54
4,38,38
5,34,34


x,各直线值,最小值
-4,"[-5, 12, -7]",-7
-3,"[-3, 11, -5]",-5
-2,"[-1, 10, -3]",-3
-1,"[1, 9, -1]",-1
0,"[3, 8, 1]",1
1,"[5, 7, 3]",3
2,"[7, 6, 5]",5
3,"[9, 5, 7]",5
4,"[11, 4, 9]",4
5,"[13, 3, 11]",3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from random import Random
from itertools import combinations
rng=Random(145)
def squared_cost(a):
    prefix=[0]
    for x in a: prefix.append(prefix[-1]+x)
    return [[(prefix[r]-prefix[l])**2 if r>=l else 0 for r in range(len(a)+1)] for l in range(len(a)+1)]
for n in range(1,16):
    for _ in range(8):
        a=[rng.randrange(6) for _ in range(n)]; cost=squared_cost(a)
        for k in range(1,n+1): assert divide_conquer_dp(cost,k)==partition_dp_quadratic(cost,k)
for n in range(1,8):
    a=list(range(1,n+1)); cost=squared_cost(a)
    for k in range(1,n+1):
        expected=min(sum(cost[l][r] for l,r in zip((0,)+cuts,cuts+(n,))) for cuts in combinations(range(1,n),k-1))
        assert partition_dp_quadratic(cost,k)==expected
try: divide_conquer_dp(squared_cost([1,1,-2]),2)
except ValueError: pass
else: raise AssertionError('failed Monge condition must be reported')
lines=[]; tree=LiChaoMin(-30,30)
for _ in range(70):
    line=(rng.randrange(-20,21),rng.randrange(-100,101)); lines.append(line); tree.add_line(*line)
    for x in range(-30,31): assert tree.query(x)==min(_evaluate(line,x) for line in lines)
for _ in range(70):
    a=[rng.randrange(-20,21) for _ in range(14)]; b=[rng.randrange(-6,7) for _ in a]
    assert cht_dp_li_chao(a,b)==cht_dp_reference(a,b)
assert cht_dp_li_chao([10**30,-10**30,10**30+1],[10**30,10**30,-2])==cht_dp_reference([10**30,-10**30,10**30+1],[10**30,10**30,-2])
print('N145: 所有本章断言通过')

N145: 所有本章断言通过


## 练习：先独立完成

**练习 1：** Knuth只在满足条件的二路合并模型使用，不推广任意K。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 寻找不满足Monge的反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1478. Allocate Mailboxes（extension）
- 1000. Minimum Cost to Merge Stones（extension）：[官方入口](https://leetcode.com/problems/minimum-cost-to-merge-stones/)
- 2463. Minimum Total Distance Traveled（comparison）：[官方入口](https://leetcode.com/problems/minimum-total-distance-traveled/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。